# SIFE-LDM V2 — Kaggle Training Notebook

**Architecture:** Complex-field diffusion with Phase-Routed MoE, Token Merging, Learnable Meta-Physics, Continuous-time SDE, and Phase-Coherent CFG conditioning.

**Dataset:** CIFAR-100 (100-class, 32×32 images)

**Outputs:** Class-conditional image generation via `cfg_guided_sample()`

## Cell 1: Environment Setup & Workspace Sync

In [ ]:
import os
import sys
import jax

print(f"Python: {sys.version}")
print(f"JAX:    {jax.__version__}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from pre-allocating all GPU memory
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'

WORKING_DIR = '/kaggle/working'
is_kaggle = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if is_kaggle:
    # Find the uploaded project dataset
    project_input_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train_vision.py' in files or 'train.py' in files:
            project_input_path = root
            break

    if project_input_path:
        print(f"Found project at: {project_input_path}")
        !cp -rfv {project_input_path}/* {WORKING_DIR}/
        %cd {WORKING_DIR}
        print("Workspace synced.")
    else:
        print("ERROR: Project not found in /kaggle/input.")
        print("Add your SIFE-LDM dataset to this notebook first.")
else:
    print("Running locally — skipping Kaggle sync.")

# Verify V2 files are present
print("\n--- V2 Architecture Check ---")
!grep -l 'LabelEncoder\|EulerMaruyamaSampler\|PhasePool\|predict_meta_physics' sife/model.py sife/diffusion.py sife/unet.py 2>/dev/null

## Cell 2: CIFAR-100 Data Download

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

!python scripts/download_cifar.py

import numpy as np
images = np.load('./data/cifar100/train_images.npy')
labels = np.load('./data/cifar100/train_labels.npy')
print(f"Dataset loaded: {images.shape} images, {labels.shape} labels")
print(f"Image range: [{images.min():.3f}, {images.max():.3f}]")

## Cell 3: V2 Training

**What's different from V1:**
- `ImageEncoder` replaces the arctan2 pixel hack → real learned RGB→complex projection
- `LabelEncoder` injects CIFAR-100 class labels as complex context embeddings
- 15% CFG dropout during training → enables class-conditional generation at inference
- `predict_meta_physics()` dynamically scales Hamiltonian params per-batch
- `PhasePool/PhaseUnpool` token merging in the transformer (50% FLOPs reduction)

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'
os.environ['JAX_TRACEBACK_FILTERING'] = 'off'

# Run V2 training
# On Kaggle GPU: ~2-4 hours for 50,000 steps (vs days on CPU)
!python scripts/train_vision.py

## Cell 4: Monitor Training Loss

Expected loss curve:
- Step 0: ~6.0 (random noise)
- Step 1000: ~2.0–3.5
- Step 10000: ~1.2–1.8
- Step 50000: ~0.8–1.2 (good generative quality)

In [ ]:
# Parse and plot the training log if available
import matplotlib.pyplot as plt
import re

# If you redirected stdout to a log file, parse it here
# Otherwise just re-run training in the cell above and read live output
log_file = 'training_log.txt'
if os.path.exists(log_file):
    steps, losses = [], []
    with open(log_file) as f:
        for line in f:
            m = re.search(r'Step (\d+): Loss = ([\d.]+)', line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))

    plt.figure(figsize=(12, 4))
    plt.plot(steps, losses, 'b-', linewidth=1.5, label='Diffusion Loss')
    plt.axhline(y=2.0, color='g', linestyle='--', alpha=0.5, label='Good quality threshold')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('SIFE-LDM V2 Training Curve (CIFAR-100)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('No log file found. Training output is inline above.')

## Cell 5: Generate Images (Unconditional)

Samples from pure noise using the `EulerMaruyamaSampler` with attractor-based early stopping.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

!python scripts/generate_images.py \
    --checkpoint checkpoints/vision/final_vision_model.pkl \
    --num_images 16 \
    --num_steps 100 \
    --output_dir ./generated_images \
    --attractor_stop

# Display the grid
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('./generated_images/generated_unconditional.png')
plt.figure(figsize=(14, 14))
plt.imshow(img)
plt.axis('off')
plt.title('SIFE-LDM V2 — Unconditional Generation (16 samples)', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 6: Class-Conditional Generation (CFG)

Uses `LabelEncoder` + `cfg_guided_sample()` to steer the SIFE field toward specific CIFAR-100 classes.

**Guidance scale:** Higher = more class-faithful, less variety. 7.5 is a good default.

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
os.environ['PYTHONPATH'] = '.'

# CIFAR-100 classes to generate
CLASSES_TO_GENERATE = {
    69: 'rocket',
    3:  'bear',
    55: 'otter',
    35: 'girl',
}

fig, axes = plt.subplots(1, len(CLASSES_TO_GENERATE), figsize=(16, 5))

for ax, (class_id, class_name) in zip(axes, CLASSES_TO_GENERATE.items()):
    !python scripts/generate_images.py \
        --checkpoint checkpoints/vision/final_vision_model.pkl \
        --class_id {class_id} \
        --guidance_scale 7.5 \
        --num_images 4 \
        --num_steps 100 \
        --output_dir ./generated_images

    img_path = f'./generated_images/generated_class{class_id}_{class_name}.png'
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f'[{class_id}] {class_name}', fontsize=12)
    ax.axis('off')

fig.suptitle('SIFE-LDM V2 — Class-Conditional CFG Generation (s=7.5)', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 7: Attractor Efficiency Benchmark

Measures how many SDE steps the attractor early-stopping saves per class — the key SIFE physics advantage.

In [ ]:
import sys, os, pickle, jax, jax.numpy as jnp
os.environ['PYTHONPATH'] = '.'
sys.path.insert(0, '.')

from sife.model import SIFELDM, SIFELDMConfig, LabelEncoder, create_model
from sife.field import SIFEConfig
from sife.diffusion import DiffusionConfig, GaussianDiffusion, EulerMaruyamaSampler
from sife.multiscale import create_multiscale_config

# Load checkpoint
with open('checkpoints/vision/final_vision_model.pkl', 'rb') as f:
    data = pickle.load(f)
params = data.params if hasattr(data, 'params') else data

config = SIFELDMConfig(
    sife=SIFEConfig(), diffusion=DiffusionConfig(num_timesteps=1000),
    multiscale=create_multiscale_config(num_levels=3, base_features=64),
    is_image=True, image_size=(32, 32), embed_dim=128, batch_size=4
)
key = jax.random.PRNGKey(42)
model, _ = create_model(config, key)
diffusion = GaussianDiffusion(DiffusionConfig(num_timesteps=1000))
sampler = EulerMaruyamaSampler(diffusion)

def model_fn(x, t, context=None):
    return model.apply(params, x, t, context=context, deterministic=True)

# Run 5 samples and measure steps saved
shape = (4, 32, 32, 128)
results = []
for trial in range(5):
    key, subkey = jax.random.split(key)
    _, steps_used = sampler.sample(
        model_fn, shape, subkey,
        num_steps=100,
        sife_config=config.sife,
        stability_threshold=0.5
    )
    pct_saved = (100 - steps_used) / 100 * 100
    results.append((steps_used, pct_saved))
    print(f"Trial {trial+1}: {steps_used}/100 steps used → {pct_saved:.1f}% saved")

avg_steps = sum(r[0] for r in results) / len(results)
avg_saved = sum(r[1] for r in results) / len(results)
print(f"\nAverage: {avg_steps:.1f} steps, {avg_saved:.1f}% compute saved via attractor stopping")

## Cell 8: Save Output & Cleanup

In [ ]:
import shutil, os

# Zip all generated images for easy download
shutil.make_archive('sife_v2_generated', 'zip', './generated_images')
print("Images zipped to sife_v2_generated.zip — download from Kaggle output panel.")

# Optional: clean up intermediate checkpoints to save disk space
import glob
intermediate = glob.glob('checkpoints/vision/checkpoint_*.pkl')
for f in intermediate:
    os.remove(f)
print(f"Cleaned {len(intermediate)} intermediate checkpoints. Final model preserved.")